# Silver Customers

**Autor:** Daniel Guzmán  
**Actividad:** Actividad 04 — Arquitectura Medallón  
**Capa:** Silver  
**Entorno:** Databricks

In [0]:
from pyspark.sql import functions as F

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

df_bronze = spark.read.table("workspace.default.bronze_customers")

print(f"Registros en Bronze: {df_bronze.count()}")
df_bronze.printSchema()
display(df_bronze.limit(5))

In [0]:
df_silver = (
    df_bronze
    .withColumnRenamed("Index", "index")
    .withColumnRenamed("Customer Id", "customer_id")
    .withColumnRenamed("First Name", "first_name")
    .withColumnRenamed("Last Name", "last_name")
    .withColumnRenamed("Company", "company")
    .withColumnRenamed("City", "city")
    .withColumnRenamed("Country", "country")
    .withColumnRenamed("Phone 1", "phone_1")
    .withColumnRenamed("Phone 2", "phone_2")
    .withColumnRenamed("Email", "email")
    .withColumnRenamed("Subscription Date", "subscription_date")
    .withColumnRenamed("Website", "website")
)

df_silver.printSchema()
display(df_silver.limit(5))

In [0]:
nulos_antes = df_silver.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
    for c in df_silver.columns
])

display(nulos_antes)

In [0]:
registros_antes = df_silver.count()

df_silver = (
    df_silver
    # Eliminar registros sin customer_id porque es la clave del cliente
    .dropna(subset=["customer_id"])
    
    # Eliminar registros sin país porque Gold agrupa por country
    .filter(F.col("country").isNotNull())
    .filter(F.trim(F.col("country")) != "")
    
    # Reemplazar ciudad vacía o nula por unknown
    .withColumn(
        "city",
        F.when(F.col("city").isNull() | (F.trim(F.col("city")) == ""), "unknown")
         .otherwise(F.trim(F.col("city")))
    )
    
    # Limpiar espacios en columnas de texto
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("first_name", F.trim(F.col("first_name")))
    .withColumn("last_name", F.trim(F.col("last_name")))
    .withColumn("company", F.trim(F.col("company")))
    .withColumn("country", F.upper(F.trim(F.col("country"))))
    .withColumn("email", F.trim(F.col("email")))
    .withColumn("website", F.trim(F.col("website")))
    
    # Convertir fecha a tipo date
    .withColumn("subscription_date", F.to_date(F.col("subscription_date")))
    
    # Eliminar duplicados por customer_id
    .dropDuplicates(["customer_id"])
    
    # Eliminar columna técnica index
    .drop("index")
)

registros_despues = df_silver.count()

print(f"Registros antes de Silver: {registros_antes}")
print(f"Registros después de Silver: {registros_despues}")
print(f"Registros eliminados: {registros_antes - registros_despues}")

df_silver.printSchema()
display(df_silver.limit(5))

In [0]:
duplicados_customer_id = (
    df_silver.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Customer_id duplicados después de limpieza: {duplicados_customer_id}")

In [0]:
nulos_despues = df_silver.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
    for c in df_silver.columns
])

display(nulos_despues)

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

spark.sql("DROP TABLE IF EXISTS silver_customers")

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_customers")

In [0]:
df_check = spark.read.table("workspace.default.silver_customers")

print(f"Registros en silver_customers: {df_check.count()}")
df_check.printSchema()
display(df_check.limit(5))